# MatchMind - Tokenization, Embeddings and Semantic Search

This notebook introduces the first Natural Language Processing and retrieval components of **MatchMind**.

The previous stages established a pipeline for loading StatsBomb data, computing deterministic football analytics, creating visualizations, and converting football possessions into natural-language documents.

This notebook focuses on the next layer:

```text
football text
      |
      v
tokenizer
      |
      v
tokens and token IDs
      |
      v
Transformer-based sentence encoder
      |
      v
embedding vector
      |
      v
cosine similarity
      |
      v
Top-K semantic retrieval
```

## Objectives

By the end of this notebook, we will be able to:

1. understand what a tokenizer does;
2. inspect tokens, token IDs, attention masks, and special tokens;
3. measure MatchMind possession lengths in tokens;
4. understand the difference between token representations and sentence embeddings;
5. generate embeddings with a pretrained Sentence Transformer;
6. implement cosine similarity manually with NumPy;
7. encode the complete possession corpus;
8. implement Top-K semantic search without a vector database;
9. combine semantic similarity with football metadata;
10. identify the limitations that motivate chunking and a vector database.

## Model

The first embedding model is:

```text
sentence-transformers/all-MiniLM-L6-v2
```

The model maps English sentences and paragraphs to a fixed-size dense vector representation. It is small enough for local experimentation and is suitable for semantic similarity and semantic search.

## Important distinction

This notebook does **not** use a generative LLM.

An embedding model does not write an answer. Its role is to transform text into vectors that can be compared for semantic similarity.

In a future RAG pipeline:

```text
retrieval -> finds relevant context
generation -> uses that context to produce an answer
```

Understanding retrieval first makes the later RAG system easier to evaluate and debug.

## 1. Project setup

The notebook is expected to live under:

```text
MatchMind/notebooks/02_text_embeddings.ipynb
```

The next cell locates the project root and adds it to Python's import path so that the notebook reuses the production code under `src/`.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()

if (cwd / "src").exists() and (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists() and (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the MatchMind project root. "
        "Start Jupyter from the MatchMind project or its notebooks directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## 2. Imports

This notebook uses:

| Library | Purpose |
| --- | --- |
| `numpy` | Vector operations and manual cosine similarity. |
| `pandas` | Tabular inspection of token lengths and retrieval results. |
| `matplotlib` | Diagnostic visualization of document lengths. |
| `transformers` | Load the tokenizer associated with the embedding model. |
| `sentence-transformers` | Load the pretrained sentence embedding model. |
| MatchMind modules | Load StatsBomb data and build possession documents. |

`sentence-transformers` is the only new direct dependency introduced for this stage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer

from src.data.statsbomb_loader import load_events
from src.text.event_formatter import build_possession_documents

print("Imports loaded successfully.")

## 3. Build the MatchMind text corpus

The corpus used in this notebook is built from the Argentina vs France 2022 World Cup final.

Each retrieval unit is currently one StatsBomb possession. A possession document contains two categories of information:

### Text

A natural-language description of the possession.

### Metadata

Structured attributes such as:

- possession ID;
- team in possession;
- period;
- start and end time;
- players;
- event types.

This separation is fundamental in RAG systems. Text can be embedded for semantic retrieval, while metadata can support filtering, traceability, and citations.

In [ ]:
MATCH_ID = 3869685

events = load_events(MATCH_ID)

possession_documents = build_possession_documents(
    events,
    include_location=False,
    include_shootout=False,
)

print(f"Raw events: {len(events)}")
print(f"Possession documents: {len(possession_documents)}")

### Inspect one possession document

Before using any NLP model, we verify what one retrieval unit contains.

An embedding can only represent the text we provide. Document construction is therefore part of retrieval design, not only preprocessing.

In [ ]:
first_document = possession_documents[0]

print("METADATA")
print("--------")
for key, value in first_document.items():
    if key != "text":
        print(f"{key}: {value}")

print("\nTEXT")
print("----")
print(first_document["text"])

## 4. Extract the text corpus

The embedding model consumes strings, so we extract the `text` field while preserving the original document list.

The lists remain aligned by index:

```text
corpus_texts[i]
      |
      | same document
      v
possession_documents[i]
```

This will later let us map a retrieved vector back to football metadata.

In [ ]:
corpus_texts = [
    document["text"]
    for document in possession_documents
]

print(f"Corpus size: {len(corpus_texts)}")
print("\nExample:")
print(corpus_texts[0])

# Part I - Tokenization

## 5. What is a tokenizer?

A Transformer does not directly process a Python string.

The text first passes through a tokenizer:

```text
raw text
   |
   v
normalization
   |
   v
subword segmentation
   |
   v
tokens
   |
   v
token IDs
   |
   v
model input
```

### Why subword tokens?

A tokenizer does not necessarily create one token per word. Rare words, names, accented forms, and punctuation may be split into smaller subword pieces.

This is useful because a fixed vocabulary can represent text that was never stored as a complete vocabulary entry.

### Token vs token ID

A token is a textual unit. A token ID is the integer index associated with that token in the model vocabulary.

The Transformer receives numerical IDs, not raw strings.

## 6. Load the tokenizer

The tokenizer must match the embedding model because tokenization is model-specific.

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer class: {tokenizer.__class__.__name__}")
print(f"Vocabulary size: {tokenizer.vocab_size}")

## 7. Inspect tokenization on a controlled football sentence

We start with a short sentence so the transformation can be inspected directly.

In [ ]:
example_text = "Kylian Mbappé takes a dangerous shot and scores for France."

tokens = tokenizer.tokenize(example_text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Original text:")
print(example_text)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

print(f"\nNumber of tokens before special tokens: {len(tokens)}")

The number of tokens does not have to equal the number of words.

This is one reason RAG chunk sizes should be discussed in **tokens**, not only characters or words.

## 8. Prepare model inputs

Calling the tokenizer directly produces the numerical structures expected by the Transformer.

Two important outputs are:

- `input_ids`: token vocabulary IDs;
- `attention_mask`: indicates which token positions contain real input rather than padding.

The tokenizer may also add architecture-specific special tokens.

In [ ]:
encoded_example = tokenizer(
    example_text,
    return_tensors="np",
)

print("Available fields:")
print(encoded_example.keys())

print("\ninput_ids shape:")
print(encoded_example["input_ids"].shape)

print("\nattention_mask shape:")
print(encoded_example["attention_mask"].shape)

print("\ninput_ids:")
print(encoded_example["input_ids"])

print("\nattention_mask:")
print(encoded_example["attention_mask"])

### Inspect the complete token sequence

Converting the full sequence of IDs back to tokenizer tokens reveals any special tokens added around the original sentence.

In [ ]:
model_input_ids = encoded_example["input_ids"][0]
model_tokens = tokenizer.convert_ids_to_tokens(model_input_ids)

print(model_tokens)
print(f"Total model input tokens: {len(model_tokens)}")

## 9. Measure possession lengths in tokens

Before choosing a chunking strategy, we measure the real token lengths of MatchMind possession documents.

Tokenization is performed without truncation so that the original document length can be observed.

In [ ]:
token_lengths = []

for text in corpus_texts:
    token_ids = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
    )["input_ids"]

    token_lengths.append(len(token_ids))

pd.Series(token_lengths, name="token_count").describe()

### Visualize the distribution

The histogram shows whether most possessions are compact or whether a small number are unusually long.

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(token_lengths, bins=20)
plt.xlabel("Tokens per possession document")
plt.ylabel("Number of possession documents")
plt.title("Distribution of Possession Document Lengths")
plt.show()

# Part II - Sentence Embeddings

## 10. Load the Sentence Transformer

The Sentence Transformer combines a Transformer encoder with pooling logic that produces one fixed-size vector for the complete input text.

The model object also exposes its maximum sequence length and embedding dimensionality.

In [ ]:
embedding_model = SentenceTransformer(MODEL_NAME)

MAX_SEQUENCE_LENGTH = embedding_model.max_seq_length
EMBEDDING_DIMENSION = embedding_model.get_sentence_embedding_dimension()

print(f"Model: {MODEL_NAME}")
print(f"Maximum sequence length: {MAX_SEQUENCE_LENGTH} tokens")
print(f"Embedding dimension: {EMBEDDING_DIMENSION}")

## 11. Compare document lengths with the model limit

Texts longer than the model's configured sequence length can be truncated during encoding.

If important football information appears after the truncation point, that information cannot influence the document embedding.

In [ ]:
length_analysis = pd.DataFrame({
    "possession_id": [d["possession_id"] for d in possession_documents],
    "team": [d["possession_team"] for d in possession_documents],
    "start_minute": [d["start_minute"] for d in possession_documents],
    "end_minute": [d["end_minute"] for d in possession_documents],
    "token_count": token_lengths,
})

length_analysis["exceeds_model_limit"] = (
    length_analysis["token_count"] > MAX_SEQUENCE_LENGTH
)

length_analysis.sort_values(
    "token_count",
    ascending=False,
).head(10)

In [ ]:
num_too_long = int(length_analysis["exceeds_model_limit"].sum())
percentage_too_long = 100 * num_too_long / len(length_analysis)

print(
    f"Documents above the model limit: "
    f"{num_too_long}/{len(length_analysis)} "
    f"({percentage_too_long:.2f}%)"
)

A document fitting inside the technical model limit does **not** automatically mean it is the best retrieval chunk.

```text
fits in context
      !=
optimal retrieval unit
```

Chunk quality must later be evaluated using retrieval results.

## 12. What is an embedding?

An embedding is a numerical vector representing information from text.

For this model:

```text
sentence or paragraph
        |
        v
Sentence Transformer
        |
        v
384-dimensional vector
```

A simplified vector looks like:

```text
[0.021, -0.114, 0.087, ..., 0.034]
```

Individual dimensions generally do not have simple human-readable meanings. Semantic information is distributed across the vector space.

### Token embeddings vs sentence embeddings

The Transformer first creates contextual representations for individual token positions. Sentence Transformers then applies pooling to obtain one fixed-size vector for the entire input text.

For our retrieval system, this final fixed-size vector is the **document embedding**.

## 13. Inspect the model architecture

Printing the model reveals the main high-level components used by Sentence Transformers.

In [ ]:
print(embedding_model)

## 14. Encode one sentence

`SentenceTransformer.encode()` performs tokenization, Transformer inference, and pooling.

In [ ]:
single_embedding = embedding_model.encode(
    example_text,
    convert_to_numpy=True,
)

print(f"Embedding type: {type(single_embedding)}")
print(f"Embedding shape: {single_embedding.shape}")
print("\nFirst 10 values:")
print(single_embedding[:10])

A short sentence and a longer paragraph are both mapped to the same fixed embedding dimensionality. This common vector space makes direct similarity comparison possible.

## 15. Controlled semantic comparison

Before searching the football corpus, we test the model on three simple sentences.

Two are semantically related football statements. The third is intentionally unrelated.

In [ ]:
comparison_texts = [
    "A France player takes a dangerous shot and scores.",
    "France creates a scoring chance that ends in a goal.",
    "The weather is sunny and warm today.",
]

comparison_embeddings = embedding_model.encode(
    comparison_texts,
    convert_to_numpy=True,
)

print(comparison_embeddings.shape)

# Part III - Cosine Similarity

## 16. What is cosine similarity?

Cosine similarity compares the direction of two vectors:

```text
cosine(a, b) = (a . b) / (||a|| * ||b||)
```

where:

- `a . b` is the dot product;
- `||a||` is the Euclidean norm of `a`;
- `||b||` is the Euclidean norm of `b`.

For semantic embeddings, a larger cosine similarity generally indicates more similar meaning.

There is no universal similarity threshold that guarantees relevance. Scores must later be evaluated in the context of the retrieval task.

## 17. Implement cosine similarity manually

We implement the formula with NumPy before using any vector database or retrieval framework.

In [ ]:
def cosine_similarity(
    vector_a: np.ndarray,
    vector_b: np.ndarray,
) -> float:
    """Compute cosine similarity between two non-zero vectors."""

    norm_a = np.linalg.norm(vector_a)
    norm_b = np.linalg.norm(vector_b)

    if norm_a == 0 or norm_b == 0:
        raise ValueError(
            "Cosine similarity is undefined for zero vectors."
        )

    return float(
        np.dot(vector_a, vector_b)
        / (norm_a * norm_b)
    )

In [ ]:
similarity_related = cosine_similarity(
    comparison_embeddings[0],
    comparison_embeddings[1],
)

similarity_unrelated = cosine_similarity(
    comparison_embeddings[0],
    comparison_embeddings[2],
)

print(
    "Related football sentences:",
    f"{similarity_related:.4f}",
)
print(
    "Football vs weather:",
    f"{similarity_unrelated:.4f}",
)

## 18. Normalized embeddings

If vectors are normalized to unit length, their Euclidean norm equals 1.

Then cosine similarity becomes equal to the dot product:

```text
cosine(a, b) = a . b
```

This is useful because large batches of similarity scores can then be computed efficiently with matrix multiplication.

In [ ]:
normalized_embeddings = embedding_model.encode(
    comparison_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

norms = np.linalg.norm(normalized_embeddings, axis=1)

manual_cosine = cosine_similarity(
    normalized_embeddings[0],
    normalized_embeddings[1],
)

dot_product = float(
    np.dot(
        normalized_embeddings[0],
        normalized_embeddings[1],
    )
)

print("Vector norms:")
print(norms)
print(f"Cosine similarity: {manual_cosine:.6f}")
print(f"Dot product:       {dot_product:.6f}")

# Part IV - Embed the MatchMind Corpus

## 19. Encode every possession

Each possession document is converted to one normalized embedding.

If:

```text
N = number of documents
D = embedding dimension
```

then the corpus embedding matrix has shape:

```text
(N, D)
```

For this small single-match corpus, the matrix can remain directly in memory as a NumPy array.

In [ ]:
corpus_embeddings = embedding_model.encode(
    corpus_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print(f"Embedding matrix shape: {corpus_embeddings.shape}")
print(f"Documents: {len(corpus_texts)}")
print(f"Embedding dimension: {corpus_embeddings.shape[1]}")

### Inspect one corpus vector

The vector is not interpreted dimension by dimension. We inspect only its representation, size, and norm.

In [ ]:
print("First possession:")
print(corpus_texts[0])

print("\nFirst 10 embedding values:")
print(corpus_embeddings[0][:10])

print(
    "\nEmbedding norm:",
    np.linalg.norm(corpus_embeddings[0]),
)

# Part V - Semantic Search

## 20. How semantic retrieval works

A query follows the same embedding pipeline as the documents:

```text
user query
    |
    v
embedding model
    |
    v
query vector
```

The query vector is then compared with every document vector.

With a small corpus, exhaustive comparison is simple:

```text
query embedding
      |
      +--> document 1 similarity
      +--> document 2 similarity
      +--> document 3 similarity
      +--> ...
```

The documents are sorted by decreasing score and the highest scoring `K` items are returned.

This is **Top-K semantic retrieval**.

## 21. Implement Top-K semantic search manually

Because both query and document embeddings are normalized, all similarity scores can be computed with one matrix-vector product:

```text
corpus_embeddings @ query_embedding
```

For a corpus matrix of shape `(N, 384)` and a query vector of shape `(384,)`, the result has shape `(N,)`.

In [ ]:
def semantic_search(
    query: str,
    documents: list[dict],
    document_embeddings: np.ndarray,
    model: SentenceTransformer,
    top_k: int = 5,
) -> pd.DataFrame:
    """Retrieve the most semantically similar possession documents."""

    if not query.strip():
        raise ValueError("The query cannot be empty.")

    if len(documents) != len(document_embeddings):
        raise ValueError(
            "The number of documents must match the number of embeddings."
        )

    if top_k <= 0:
        raise ValueError("top_k must be greater than zero.")

    query_embedding = model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    scores = document_embeddings @ query_embedding

    top_k = min(top_k, len(documents))
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):
        document = documents[int(index)]

        results.append({
            "rank": rank,
            "score": float(scores[index]),
            "possession_id": document["possession_id"],
            "team": document["possession_team"],
            "period": document["period"],
            "start_minute": document["start_minute"],
            "end_minute": document["end_minute"],
            "event_types": document["event_types"],
            "text": document["text"],
        })

    return pd.DataFrame(results)

## 22. First semantic query

The query does not need to use the exact wording present in the documents.

This is the key difference between semantic retrieval and exact keyword matching.

In [ ]:
query = "A dangerous France attack ending with a shot"

results = semantic_search(
    query=query,
    documents=possession_documents,
    document_embeddings=corpus_embeddings,
    model=embedding_model,
    top_k=5,
)

results[
    [
        "rank",
        "score",
        "possession_id",
        "team",
        "start_minute",
        "end_minute",
        "event_types",
    ]
]

### Inspect retrieved documents

A numerical similarity score is not enough to determine retrieval quality.

The retrieved text must be inspected to determine whether it actually answers the intended information need.

In [ ]:
for _, result in results.iterrows():
    print("=" * 100)
    print(
        f"Rank {int(result['rank'])} | "
        f"Score: {result['score']:.4f} | "
        f"Possession: {result['possession_id']} | "
        f"Team: {result['team']}"
    )
    print("-" * 100)
    print(result["text"])
    print()

## 23. Test several semantic queries

Testing different query formulations helps reveal both the strengths and weaknesses of the representation and model.

In [ ]:
test_queries = [
    "France creates a dangerous attack and takes a shot",
    "Argentina creates a chance for Lionel Messi",
    "A player wins the ball back and starts an attack",
    "A sequence containing several passes",
]

for test_query in test_queries:
    print("=" * 100)
    print(f"QUERY: {test_query}")
    print("=" * 100)

    query_results = semantic_search(
        query=test_query,
        documents=possession_documents,
        document_embeddings=corpus_embeddings,
        model=embedding_model,
        top_k=3,
    )

    for _, result in query_results.iterrows():
        print(
            f"{int(result['rank'])}. "
            f"score={result['score']:.4f} | "
            f"team={result['team']} | "
            f"minute={result['start_minute']} | "
            f"possession={result['possession_id']}"
        )

    print()

# Part VI - Metadata-Assisted Retrieval

## 24. Why vector similarity alone is not enough

The embedding model captures semantic similarity, but a high score does not guarantee that all structured constraints are satisfied.

For example, a query may explicitly require:

```text
team = France
event type = Shot
minute >= 70
```

Those conditions are already available as metadata and do not need to be inferred from vector similarity.

A future retrieval system can therefore combine:

```text
metadata filtering
        +
semantic similarity
```

## 25. Filter candidates before vector ranking

For this experiment, we keep only France possessions that contain a shot and then run semantic ranking on that subset.

In [ ]:
candidate_indices = [
    index
    for index, document in enumerate(possession_documents)
    if (
        document["possession_team"] == "France"
        and "Shot" in document["event_types"]
    )
]

filtered_documents = [
    possession_documents[index]
    for index in candidate_indices
]

filtered_embeddings = corpus_embeddings[candidate_indices]

print(
    "France possessions containing a shot:",
    len(filtered_documents),
)

In [ ]:
filtered_results = semantic_search(
    query="A dangerous attack that produces a goal-scoring chance",
    documents=filtered_documents,
    document_embeddings=filtered_embeddings,
    model=embedding_model,
    top_k=5,
)

filtered_results[
    [
        "rank",
        "score",
        "possession_id",
        "team",
        "start_minute",
        "event_types",
    ]
]

This experiment anticipates an important vector-database feature:

```text
filter candidate documents using metadata
                |
                v
compare embedding vectors
                |
                v
return ranked semantic matches
```

# Part VII - Interpretation and Next Steps

## 26. What we have learned

The MatchMind retrieval pipeline now reaches:

```text
StatsBomb events
      |
      v
possession documents
      |
      v
text
      |
      v
tokenizer
      |
      v
tokens and token IDs
      |
      v
Sentence Transformer
      |
      v
384-dimensional embeddings
      |
      v
cosine similarity
      |
      v
Top-K semantic retrieval
```

No generative model is involved yet.

## 27. Core concepts

### Token
A model-specific textual unit produced by a tokenizer.

### Token ID
The integer vocabulary identifier associated with a token.

### Attention mask
A model input indicating which positions contain real input tokens rather than padding.

### Transformer
The neural architecture that produces contextual representations from tokenized text.

### Sentence embedding
A fixed-size numerical representation of an entire sentence or paragraph.

### Embedding dimension
The number of components in the vector representation.

### Cosine similarity
A similarity measure based on the angle between two vectors.

### Semantic search
Retrieval based on similarity of meaning rather than only exact word overlap.

### Top-K retrieval
Returning the `K` highest scoring candidate documents.

### Metadata
Structured information stored alongside a document, such as team, time, possession ID, players, and event types.

## 28. Current limitations

The current system is intentionally simple.

It does not yet include:

- a formal chunking strategy;
- persistent embedding storage;
- a vector database;
- approximate nearest-neighbor indexing;
- sparse retrieval such as BM25;
- hybrid search;
- reranking;
- a retrieval evaluation dataset;
- Recall@K or MRR;
- a generative LLM;
- RAG prompting;
- an AI agent.

The complete corpus is embedded in memory and searched exhaustively with NumPy. This is appropriate for one match and for learning, but not for a large production corpus.

## 29. Why chunking is the next question

This notebook used:

```text
one possession = one retrieval document
```

That is a football-aware starting point, but it is still a hypothesis.

A good retrieval chunk should ideally:

- contain enough context to answer a useful question;
- avoid mixing unrelated information;
- remain inside the model input limit;
- produce discriminative embeddings;
- preserve metadata and traceability.

Possible strategies include:

```text
event-level chunks
possession-level chunks
long possessions split into smaller chunks
multiple related possessions per chunk
fixed token windows with overlap
```

The next stage will treat chunking as an engineering and evaluation problem rather than selecting an arbitrary character count.

## 30. Next project stage

Once the experiments in this notebook are understood, stable embedding and retrieval logic can be moved into production modules such as:

```text
src/
├── embeddings/
│   └── encoder.py
│
└── retrieval/
    └── semantic_search.py
```

After that, the project can progress toward:

```text
chunking experiments
        |
        v
vector database
        |
        v
retrieval evaluation
        |
        v
RAG generation
        |
        v
agentic AI
```

The same development principle remains in place:

```text
learn -> experiment -> modularize -> test -> commit
```

## References

- Sentence Transformers documentation: https://www.sbert.net/
- Hugging Face model: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
- Hugging Face tokenizer documentation: https://huggingface.co/docs/transformers/main_classes/tokenizer

The semantic search implementation in this notebook is intentionally written with NumPy so that vector comparison is understood before introducing a vector database.